In [2]:
"""
FINAL ANALYSIS DASHBOARD
------------------------
1. Loads data from SQLite.
2. Performs Scalar Analysis (Logistic Regression, Feature Importance).
3. Performs Token-Level Analysis (Identifying specific words that confuse the model).
4. Generates rich, interactive visualizations.
"""

import sqlite3
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, confusion_matrix, roc_curve, classification_report
from IPython.display import display, Markdown, HTML

# --- CONFIGURATION ---
DB_PATH = 'db/results.sqlite'
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams['figure.figsize'] = (10, 6)

# ==============================================================================
# 1. DATA LOADING & PREPROCESSING
# ==============================================================================

def load_and_prep_data(db_path):
    print(f"Loading data from {db_path}...")
    conn = sqlite3.connect(db_path)
    
    # Load Main Results
    query = """
    SELECT 
        question_text, 
        is_correct as Correct,
        uq_avg_entropy as Entropy, 
        uq_min_logit_gap as LogitGap, 
        uq_heuristic_score as Heuristic, 
        uq_mech_score as Mechanistic,
        gen_len as Length,
        uq_tokens, uq_entropy_trace, uq_logit_gap_trace
    FROM Results
    WHERE eval_method != 'Pending'
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Helper to parse JSON columns
    def safe_parse(val):
        try:
            return json.loads(val)
        except (TypeError, json.JSONDecodeError):
            return []

    # Parse JSON Traces for Token Analysis
    for col in ['uq_tokens', 'uq_entropy_trace', 'uq_logit_gap_trace']:
        df[col] = df[col].apply(safe_parse)

    print(f"✅ Loaded {len(df)} records.")
    return df

# ==============================================================================
# 2. SCALAR ANALYSIS (MACRO VIEW)
# ==============================================================================

def analyze_scalar_drivers(df):
    display(Markdown("## 📊 Part 1: Macro Analysis (Scalar Drivers)"))
    display(Markdown("Which high-level metrics best predict if the model is wrong?"))

    features = ['Entropy', 'LogitGap', 'Heuristic', 'Mechanistic', 'Length']
    
    # Prepare Data
    df_clean = df.copy().dropna(subset=features)
    X = df_clean[features]
    y = df_clean['Correct']
    
    # Train/Test Split (stratified by correctness)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Logistic Regression
    clf = LogisticRegression(class_weight='balanced', penalty='l2', C=1.0)
    clf.fit(X_train_s, y_train)
    probs = clf.predict_proba(X_test_s)[:, 1]

    # --- A. FEATURE IMPORTANCE TABLE ---
    coef_df = pd.DataFrame({
        'Feature': features,
        'Weight (Log-Odds)': clf.coef_[0],
        'Odds Ratio': np.exp(clf.coef_[0]),
        'Abs Impact': np.abs(clf.coef_[0])
    }).sort_values('Abs Impact', ascending=False)

    display(Markdown("### 1. Feature Importance (Odds Ratios)"))
    display(
        coef_df.style.background_gradient(cmap='coolwarm', subset=['Weight (Log-Odds)'])
        .format("{:.4f}", subset=['Weight (Log-Odds)', 'Odds Ratio', 'Abs Impact'])
    )

    # --- B. PERFORMANCE METRICS ---
    auc = roc_auc_score(y_test, probs)
    brier = brier_score_loss(y_test, probs)
    
    metrics_df = pd.DataFrame({
        'Metric': ['ROC-AUC', 'Brier Score (Calib)', 'Log Loss'],
        'Value': [auc, brier, log_loss(y_test, probs)]
    })
    
    display(Markdown("### 2. Model Performance"))
    display(metrics_df.style.hide(axis='index').format({"Value": "{:.4f}"}))

    # --- C. PLOTS ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Weights Plot
    colors = ['forestgreen' if x > 0 else 'crimson' for x in coef_df['Weight (Log-Odds)']]
    sns.barplot(x='Weight (Log-Odds)', y='Feature', data=coef_df, palette=colors, ax=ax1)
    ax1.set_title("Feature Weights (Direction & Magnitude)")
    ax1.axvline(0, color='black', lw=1)

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc:.2f}')
    ax2.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax2.set_title("ROC Curve")
    ax2.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

# ==============================================================================
# 3. TOKEN ANALYSIS (MICRO VIEW)
# ==============================================================================

def analyze_token_drivers(df):
    display(Markdown("## 🔍 Part 2: Micro Analysis (Token Attribution)"))
    display(Markdown("Identifying specific *generated* tokens that cause uncertainty spikes."))

    # Explode Data: Create 1 row per GENERATED token
    token_rows = []
    for idx, row in df.iterrows():
        # Sync lengths
        limit = min(len(row['uq_tokens']), len(row['uq_entropy_trace']))
        for i in range(limit):
            token_rows.append({
                'token': str(row['uq_tokens'][i]),
                'entropy': row['uq_entropy_trace'][i],
                'logit_gap': row['uq_logit_gap_trace'][i] if row['uq_logit_gap_trace'] else 0,
                'is_correct': row['Correct'],
                'pos': i
            })
    
    token_df = pd.DataFrame(token_rows)
    
    # --- A. THE "CONFUSION" DICTIONARY ---
    # Filter for tokens appearing at least 15 times to ensure statistical relevance
    stats = token_df.groupby('token').agg({
        'entropy': 'mean',
        'logit_gap': 'mean',
        'token': 'count'
    }).rename(columns={'token': 'count'})
    
    confusion = stats[stats['count'] >= 15].sort_values('entropy', ascending=False).head(10)
    clarity   = stats[stats['count'] >= 15].sort_values('entropy', ascending=True).head(10)

    display(Markdown("### 1. Most Uncertain Tokens (High Entropy)"))
    display(confusion.style.background_gradient(cmap='Reds', subset=['entropy']))

    display(Markdown("### 2. Most Certain Tokens (Low Entropy)"))
    display(clarity.style.background_gradient(cmap='Greens', subset=['entropy']))

    # --- B. UNCERTAINTY OVER TIME (TRACE) ---
    plt.figure(figsize=(12, 5))
    # Compare traces for Correct vs Incorrect answers
    sns.lineplot(data=token_df[token_df['pos'] < 100], x='pos', y='entropy', hue='is_correct', palette={0: 'crimson', 1: 'forestgreen'})
    plt.title("Average Entropy Trace: Correct vs. Incorrect Answers")
    plt.xlabel("Token Position")
    plt.ylabel("Entropy")
    plt.legend(title="Is Answer Correct?", labels=['Incorrect', 'Correct'])
    plt.show()

    return token_df

# ==============================================================================
# 4. EXECUTION
# ==============================================================================

# Run the pipeline
try:
    df = load_and_prep_data(DB_PATH)
    
    if not df.empty:
        analyze_scalar_drivers(df)
        token_df = analyze_token_drivers(df)
        
        display(Markdown("✅ **Analysis Complete.**"))
    else:
        display(Markdown("⚠️ Database found but table is empty."))

except Exception as e:
    display(Markdown(f"❌ **Error:** {str(e)}"))
    print("Ensure 'db/results.sqlite' exists and has the correct schema.")

Loading data from db/results.sqlite...
✅ Loaded 573 records.


## 📊 Part 1: Macro Analysis (Scalar Drivers)

Which high-level metrics best predict if the model is wrong?

❌ **Error:** This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)

Ensure 'db/results.sqlite' exists and has the correct schema.
